# Oligopoly & Nash Equilibria in Trading Strategies

Two classic results from market microstructure, both genuinely game-theoretic:

1. **N-insider Nash equilibrium** (Kyle, 1985 -> Holden & Subrahmanyam, 1992):
   what happens when several traders all have the *same* private information
   and compete against each other, instead of one monopolist insider?
2. **Cournot competition among market makers**: dealers competing on how much
   liquidity (depth) to supply, with the bid-ask spread as the market-clearing
   "price."

Both models are solved in closed form in `oligopoly_games/`, and both are
checked against Monte Carlo simulation here.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from oligopoly_games.kyle_model import solve_kyle
from oligopoly_games.nash_insiders import solve_nash_insiders, competition_curve
from oligopoly_games.cournot_liquidity import solve_cournot_symmetric, spread_vs_n_curve
from oligopoly_games.backtest import simulate_nash_insiders, simulate_cournot_entry

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


## 1. The baseline: a single monopolist insider (Kyle, 1985)

One informed trader observes the asset's true value $v \sim N(0, \sigma_v^2)$.
Noise traders submit random order flow $u \sim N(0, \sigma_u^2)$. A competitive
market maker sees only the *total* order flow and prices at

$$p = \lambda(x + u)$$

The insider's optimal linear strategy and the market maker's pricing rule solve to:

$$\beta = \frac{\sigma_u}{\sigma_v}, \qquad \lambda = \frac{\sigma_v}{2\sigma_u}, \qquad \mathbb{E}[\text{profit}] = \tfrac{1}{2}\sigma_v\sigma_u$$


In [ ]:
kyle = solve_kyle(sigma_v=2.0, sigma_u=3.0)
print(f"beta   = {kyle.beta:.4f}")
print(f"lambda = {kyle.lam:.4f}")
print(f"expected insider profit = {kyle.expected_profit:.4f}")


## 2. What happens with $N$ insiders sharing the *same* signal?

Now suppose $N$ traders all observe the same $v$ and compete simultaneously.
Solving the symmetric Nash equilibrium (full derivation in
`nash_insiders.py`) gives:

$$\beta(N) = \frac{\sigma_u}{\sigma_v\sqrt{N}}, \qquad
\lambda(N) = \frac{\sigma_v\sqrt{N}}{\sigma_u(N+1)}, \qquad
\pi_{\text{per insider}}(N) = \frac{\sigma_u\sigma_v}{\sqrt{N}(N+1)}$$

At $N=1$ this collapses exactly to the Kyle monopoly result above.


In [ ]:
n1 = solve_nash_insiders(1, sigma_v=2.0, sigma_u=3.0)
print("N=1 matches monopoly Kyle exactly:",
      np.isclose(n1.beta, kyle.beta), np.isclose(n1.lam, kyle.lam))

ns = [1, 2, 3, 5, 10, 25, 50, 100]
curve = competition_curve(ns, sigma_v=2.0, sigma_u=3.0)
for n, b, l, pp, tp in zip(curve["n"], curve["beta"], curve["lambda"],
                            curve["profit_per_insider"], curve["total_profit"]):
    print(f"N={n:4d}   beta={b:.4f}   lambda={l:.4f}   profit/insider={pp:.4f}   total profit={tp:.4f}")


### The key tension: individual restraint vs. aggregate aggressiveness

Each insider individually trades *less* aggressively as $N$ grows (free-riding
on the fact that others share the same edge) -- but because there are more of
them, the *aggregate* signal-driven order flow $N\beta(N)$ still goes up, and
the price becomes more informative. Meanwhile the *total* pie of informational
rent shrinks toward zero: competition can't be stopped just because everyone
agrees to trade less.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

ax = axes[0, 0]
ax.plot(curve["n"], curve["beta"], "o-", color="#2563eb")
ax.set_title(r"Individual intensity $\beta(N)$ decreases with N")
ax.set_xlabel("N insiders"); ax.set_ylabel(r"$\beta(N)$")

ax = axes[0, 1]
ax.plot(curve["n"], curve["n"] * curve["beta"], "o-", color="#dc2626")
ax.set_title(r"Aggregate intensity $N \cdot \beta(N)$ increases with N")
ax.set_xlabel("N insiders"); ax.set_ylabel(r"$N \cdot \beta(N)$")

ax = axes[1, 0]
ax.plot(curve["n"], curve["total_profit"], "o-", color="#16a34a", label="total insider profit")
ax.plot(curve["n"], curve["profit_per_insider"], "s--", color="#16a34a", alpha=0.5, label="profit per insider")
ax.set_title("Informational rents are competed away")
ax.set_xlabel("N insiders"); ax.set_ylabel("expected profit")
ax.legend()

ax = axes[1, 1]
ax.plot(curve["n"], curve["posterior_var"], "o-", color="#9333ea")
ax.axhline(0, color="gray", lw=0.8)
ax.set_title("Price informativeness improves with N")
ax.set_xlabel("N insiders"); ax.set_ylabel("posterior Var(v | order flow)")

plt.tight_layout()
plt.show()


## 3. Does the theory hold up in simulation?

The formulas above came from a first-order-condition argument assuming a
*linear* equilibrium exists. Let's check by literally simulating the market
(draw $v$, $u$, have every insider submit $x=\beta(N) v$, form the price) tens
of thousands of times, and comparing the *empirically recovered* $\lambda$ and
profit per insider to the closed-form prediction.


In [ ]:
sim = simulate_nash_insiders(n=5, sigma_v=2.0, sigma_u=3.0, n_trials=100_000, seed=1)
print(sim)
print()
print(f"lambda: theory={sim.theoretical_lambda:.5f}  empirical={sim.empirical_lambda:.5f}")
print(f"profit: theory={sim.theoretical_profit_per_insider:.5f}  empirical={sim.empirical_profit_per_insider:.5f}")


## 4. Market makers as Cournot oligopolists

Now switch sides of the market: instead of informed traders competing, think
of $N$ market makers competing on *how much depth* to post. The more
aggregate depth $Q=\sum q_i$ they supply, the lower the spread investors are
willing to pay:

$$s(Q) = a - bQ$$

Each market maker has marginal cost $c_i$ of providing liquidity and chooses
quantity to maximize $(s(Q)-c_i)q_i$. Solving the standard $N$-firm Cournot
problem gives an equilibrium spread that converges to marginal cost as
competition increases -- exactly the textbook industrial-organization result,
here applied to bid-ask spreads instead of widget prices.


In [ ]:
a, b, c = 1.0, 0.01, 0.10
ns = [1, 2, 3, 5, 10, 20, 50, 200]
curve_c = spread_vs_n_curve(ns, a=a, b=b, c=c)

for n, s, q in zip(curve_c["n"], curve_c["spread"], curve_c["aggregate_depth"]):
    print(f"N={n:4d}   equilibrium spread={s:.5f}   aggregate depth={q:.2f}")

fig, ax = plt.subplots()
ax.plot(curve_c["n"], curve_c["spread"], "o-", color="#2563eb", label="equilibrium spread $s^*(N)$")
ax.axhline(c, color="#dc2626", ls="--", label="marginal cost $c$ (competitive limit)")
ax.set_xlabel("N market makers"); ax.set_ylabel("equilibrium spread")
ax.set_title("Spread compresses toward marginal cost as market makers enter")
ax.legend()
plt.show()


## 5. Putting both results side by side

Both models tell the same underlying story from opposite sides of the market:
**adding competitors does not eliminate the underlying friction (information
asymmetry, or the cost of providing liquidity) -- it just forces it to be
priced more competitively.** As $N\to\infty$ in both models, the market
converges toward its frictionless/competitive benchmark, but never because of
any single trader's restraint -- always because of the *number* of traders.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax = axes[0]
ns = list(range(1, 60))
profits = [solve_nash_insiders(n, 2.0, 3.0).total_insider_profit for n in ns]
ax.plot(ns, profits, color="#16a34a")
ax.set_title("Insider rents -> 0 as N grows")
ax.set_xlabel("N insiders"); ax.set_ylabel("total expected profit")

ax = axes[1]
spreads = [solve_cournot_symmetric(n, a, b, c).spread for n in ns]
ax.plot(ns, spreads, color="#2563eb")
ax.axhline(c, color="#dc2626", ls="--", label="marginal cost")
ax.set_title("Spread -> marginal cost as N grows")
ax.set_xlabel("N market makers"); ax.set_ylabel("equilibrium spread")
ax.legend()

plt.tight_layout()
plt.show()


## References

- Kyle, A. S. (1985). *Continuous Auctions and Insider Trading*. Econometrica.
- Holden, C. W., & Subrahmanyam, A. (1992). *Long-Lived Private Information and
  Imperfect Competition*. Journal of Finance.
- Standard N-firm Cournot oligopoly (any intermediate microeconomics /
  industrial organization text), applied here to liquidity provision.
